In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [17]:
from loaders._gen_binary import generate_data

import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import balanced_accuracy_score
from numpy.linalg import eigh
from sklearn.model_selection import TimeSeriesSplit

In [6]:
# ================================================================
# ClusterMetricClassifier: Cluster-based classification + Metric learning
#  - Metrics: 'identity', 'whiten_global', 'whiten_within', 'diag_fisher', 'triplet_diag'
#  - Clustering: 'per_class' (mỗi lớp có K cụm) hoặc 'global' (K cụm toàn cục)
#  - Dự đoán: gán nhãn theo tâm cụm gần nhất trong không gian đã biến đổi
# ================================================================

class ClusterMetricClassifier:
    """
    Cluster-based classifier with pluggable metric learning.
    Distance in original space is d_M(x, c) = (x - c)^T M (x - c), with M = A^T A.
    We learn A depending on `metric`.
    """
    def __init__(
        self,
        metric: str = "whiten_within",    # 'identity' | 'whiten_global' | 'whiten_within' | 'diag_fisher' | 'triplet_diag'
        cluster_mode: str = "per_class",  # 'per_class' | 'global'
        n_clusters: int = 2,              # if per_class: clusters per class; if global: total clusters
        reg: float = 1e-6,                # small Tikhonov regularization for covariance inversion
        triplet_iters: int = 3000,        # iterations for 'triplet_diag'
        triplet_lr: float = 0.05,
        triplet_alpha: float = 1e-3,      # L2 reg on weights
        random_state: int = 0,
        standardize: bool = True
    ):
        self.metric = metric
        self.cluster_mode = cluster_mode
        self.n_clusters = int(n_clusters)
        self.reg = reg
        self.triplet_iters = int(triplet_iters)
        self.triplet_lr = float(triplet_lr)
        self.triplet_alpha = float(triplet_alpha)
        self.random_state = random_state
        self.standardize = standardize

        # learned/stored
        self.scaler_ = None
        self.A_ = None                   # linear transform (d x d); for diagonal metric: diag(sqrt(w))
        self.centers_ = None             # cluster centers in transformed space (k x d)
        self.center_labels_ = None       # label per center (k,)
        self.classes_ = None

    # ----------------- helpers -----------------
    @staticmethod
    def _cov_eig_inv_sqrt(X, reg=1e-6):
        """
        Compute Sigma^{-1/2} for zero-mean X (n x d)
        """
        n, d = X.shape
        # unbiased covariance: (X^T X)/(n-1)
        C = (X.T @ X) / max(1, (n - 1))
        # regularize
        C = C + reg * np.eye(d)
        # eigen-decompose
        vals, vecs = eigh(C)
        vals = np.clip(vals, reg, None)
        D_inv_sqrt = np.diag(1.0 / np.sqrt(vals))
        A = D_inv_sqrt @ vecs.T  # Sigma^{-1/2} times U^T (but ensure A^T A = C^{-1})
        # Check: A^T A = vecs @ D_inv_sqrt^2 @ vecs^T = C^{-1}
        return A

    @staticmethod
    def _pooled_within_scatter(X, y):
        """
        Pooled within-class scatter matrix Sw = sum_c sum_{i in c} (x_i - mu_c)(x_i - mu_c)^T / (n - n_classes)
        (scaled like covariance). X assumed already zero-mean wrt global or standardized.
        """
        classes = np.unique(y)
        d = X.shape[1]
        Sw = np.zeros((d, d), dtype=float)
        denom = 0
        for c in classes:
            Xc = X[y == c]
            if Xc.shape[0] <= 1:
                continue
            muc = Xc.mean(axis=0, keepdims=True)
            Zc = Xc - muc
            Sw += (Zc.T @ Zc)
            denom += (Xc.shape[0] - 1)
        if denom <= 0:
            return np.eye(d)
        return Sw / denom

    @staticmethod
    def _between_scatter_diag(X, y):
        """
        Diagonal of between-class scatter (per-feature). Binary-friendly but works for multiclass.

        For classes c with prior pi_c and means mu_c, Sb = sum_c pi_c (mu_c - mu)(mu_c - mu)^T.
        We return only the diagonal of Sb.
        """
        n, d = X.shape
        classes, counts = np.unique(y, return_counts=True)
        pi = counts / counts.sum()
        mu = X.mean(axis=0)
        Sb_diag = np.zeros(d, dtype=float)
        for pc, c in zip(pi, classes):
            muc = X[y == c].mean(axis=0)
            diff = (muc - mu)
            Sb_diag += pc * (diff ** 2)
        return Sb_diag

    def _learn_metric(self, X, y):
        """
        Learn A so that M = A^T A is the desired metric.
        X, y are TRAIN data after (optional) standardization. X assumed roughly zero-mean.
        """
        n, d = X.shape

        if self.metric == "identity":
            A = np.eye(d)

        elif self.metric == "whiten_global":
            # A = Sigma^{-1/2}, with Sigma = Cov(X)
            A = self._cov_eig_inv_sqrt(X, reg=self.reg)

        elif self.metric == "whiten_within":
            # A = Sw^{-1/2}, pooled within-class covariance whitening (Fisher style)
            Sw = self._pooled_within_scatter(X, y)
            # regularize
            Sw = Sw + self.reg * np.eye(d)
            # eig inverse sqrt
            vals, vecs = eigh(Sw)
            vals = np.clip(vals, self.reg, None)
            A = (np.diag(1.0 / np.sqrt(vals)) @ vecs.T)

        elif self.metric == "diag_fisher":
            # Diagonal Fisher-like scaling: w_j ∝ sqrt( (Sb_jj + reg) / (Sw_jj + reg) )
            Sw = self._pooled_within_scatter(X, y)
            Sw_diag = np.clip(np.diag(Sw), self.reg, None)
            Sb_diag = self._between_scatter_diag(X, y)
            w = np.sqrt((Sb_diag + self.reg) / (Sw_diag + self.reg))
            # normalize scale to keep magnitudes reasonable
            if np.linalg.norm(w) > 0:
                w = w / (np.mean(w) + 1e-12)
            A = np.diag(np.sqrt(np.clip(w, 0.0, None)))  # A so that M = diag(w)

        elif self.metric == "triplet_diag":
            # Learn diagonal weights w >= 0 by minimizing hinge loss on triplets
            # L = max(0, 1 + d_w(x_i,x_pos) - d_w(x_i,x_neg)) + alpha ||w||^2
            # d_w(a,b) = sum_j w_j (a_j - b_j)^2
            rng = np.random.default_rng(self.random_state)
            w = np.ones(d, dtype=float)

            # pre-group indices
            classes = np.unique(y)
            idx_by_class = {c: np.where(y == c)[0] for c in classes}
            all_idx = np.arange(n)

            for t in range(self.triplet_iters):
                # sample anchor
                i = int(rng.integers(0, n))
                yi = y[i]
                # positive
                pos_candidates = idx_by_class[yi]
                if pos_candidates.size <= 1:
                    continue
                j = int(rng.choice(pos_candidates))
                while j == i:
                    j = int(rng.choice(pos_candidates))
                # negative
                neg_class = int(rng.choice([c for c in classes if c != yi]))
                k = int(rng.choice(idx_by_class[neg_class]))

                diff_pos = (X[i] - X[j]) ** 2   # d-dim
                diff_neg = (X[i] - X[k]) ** 2

                dpos = np.dot(w, diff_pos)
                dneg = np.dot(w, diff_neg)

                margin = 1.0 + dpos - dneg
                # gradient
                if margin > 0:
                    grad = diff_pos - diff_neg + 2 * self.triplet_alpha * w
                else:
                    grad = 2 * self.triplet_alpha * w

                # sgd step
                w -= self.triplet_lr * grad
                # project to nonnegative
                w = np.maximum(w, 0.0)
                # optional normalization to prevent blow-up / collapse
                if (t + 1) % 200 == 0:
                    m = np.mean(w[w > 0]) if np.any(w > 0) else 1.0
                    w /= (m + 1e-12)

            A = np.diag(np.sqrt(w))  # so that M = diag(w)

        else:
            raise ValueError(f"Unknown metric: {self.metric}")

        return A

    # ----------------- API -----------------
    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        # 1) standardize (recommended for stable covariance/optim)
        if self.standardize:
            self.scaler_ = StandardScaler()
            Xs = self.scaler_.fit_transform(X)
        else:
            self.scaler_ = None
            Xs = X

        # 2) learn metric A
        self.A_ = self._learn_metric(Xs, y)

        # 3) transform data
        Z = Xs @ self.A_.T   # (n x d)

        rng = np.random.RandomState(self.random_state)
        self.classes_ = np.unique(y)

        centers = []
        labels  = []

        if self.cluster_mode == "per_class":
            # K-means inside each class, attach class label to its centers
            for c in self.classes_:
                Zc = Z[y == c]
                k = min(self.n_clusters, max(1, Zc.shape[0]))
                km = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
                km.fit(Zc)
                centers.append(km.cluster_centers_)
                labels.append(np.full(k, c))
            self.centers_ = np.vstack(centers)
            self.center_labels_ = np.concatenate(labels)

        elif self.cluster_mode == "global":
            k = min(self.n_clusters, max(1, Z.shape[0]))
            km = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
            km.fit(Z)
            self.centers_ = km.cluster_centers_

            # Majority vote label per cluster
            # Assign each training point to nearest center (already in km.labels_)
            maj = []
            for cluster_id in range(k):
                idx = np.where(km.labels_ == cluster_id)[0]
                if idx.size == 0:
                    # empty cluster (unlikely), assign most frequent class globally
                    maj.append(self.classes_[np.argmax([(y == c).sum() for c in self.classes_])])
                else:
                    vals, cnts = np.unique(y[idx], return_counts=True)
                    maj.append(vals[np.argmax(cnts)])
            self.center_labels_ = np.array(maj)

        else:
            raise ValueError(f"Unknown cluster_mode: {self.cluster_mode}")

        return self

    def _transform(self, X):
        X = np.asarray(X, dtype=float)
        if self.scaler_ is not None:
            X = self.scaler_.transform(X)
        return X @ self.A_.T

    def predict(self, X):
        Z = self._transform(X)
        # nearest center in transformed Euclidean
        # returns (n_samples,)
        # compute pairwise squared distances to centers
        # dist^2 = ||z||^2 + ||c||^2 - 2 z·c
        z2 = np.sum(Z * Z, axis=1, keepdims=True)             # (n,1)
        c2 = np.sum(self.centers_ * self.centers_, axis=1)    # (k,)
        dot = Z @ self.centers_.T                             # (n,k)
        d2  = z2 + c2[None, :] - 2 * dot
        nn  = np.argmin(d2, axis=1)
        return self.center_labels_[nn]

    def score_balanced_accuracy(self, X, y):
        y_pred = self.predict(X)
        return balanced_accuracy_score(y, y_pred)

In [ ]:
bacc = {
    "identity": [],
    "whiten_global": [],
    "whiten_within": [],
    "diag_fisher": [],
    "triplet_diag": []
}

metrics = list(bacc.keys())
for seed in range(30):
    pack = generate_data(seed=seed)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    # Try different metrics:
    for m in metrics:
        clf = ClusterMetricClassifier(metric=m, cluster_mode="per_class", n_clusters=5, random_state=0)
        clf.fit(X_train, y_train)
        test_bacc = clf.score_balanced_accuracy(X_test, y_test)
        # print(f"Metric {m} | Test BA: {test_bacc}")
        bacc[m].append(test_bacc)

for m in metrics:
    arr = np.array(bacc[m])
    print(f"Metric {m} | Mean Test BA: {arr.mean():.4f} | Std: {arr.std():.4f}")

Metric identity | Mean Test BA: 0.5799 | Std: 0.0287
Metric whiten_global | Mean Test BA: 0.5900 | Std: 0.0241
Metric whiten_within | Mean Test BA: 0.5920 | Std: 0.0230
Metric diag_fisher | Mean Test BA: 0.5873 | Std: 0.0221
Metric triplet_diag | Mean Test BA: 0.5812 | Std: 0.0474


In [18]:
# ================================================================
# Cluster-Metric Ensemble (CME):
#   - Base learner: cluster-based classifier with learned metric (no NCA).
#   - Ensemble over diverse metrics + random seeds (+ optional feature subspace).
#   - Weights learned via TimeSeriesSplit CV using balanced accuracy.
#   - Experiment default: n_clusters=5 (per_class).
# ================================================================

# ----------------- Base metric utilities -----------------
def _cov_eig_inv_sqrt(X, reg=1e-6):
    """Return A = Sigma^{-1/2} with Tikhonov regularization."""
    n, d = X.shape
    C = (X.T @ X) / max(1, (n - 1))
    C = C + reg * np.eye(d)
    vals, vecs = eigh(C)
    vals = np.clip(vals, reg, None)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(vals))
    A = D_inv_sqrt @ vecs.T  # ensures A^T A = C^{-1}
    return A

def _pooled_within_scatter(X, y):
    classes = np.unique(y)
    d = X.shape[1]
    Sw = np.zeros((d, d), float)
    denom = 0
    for c in classes:
        Xc = X[y == c]
        if Xc.shape[0] <= 1:
            continue
        muc = Xc.mean(0, keepdims=True)
        Zc = Xc - muc
        Sw += Zc.T @ Zc
        denom += (Xc.shape[0] - 1)
    if denom <= 0:  # fallback
        return np.eye(d)
    return Sw / denom

def _between_scatter_diag(X, y):
    n, d = X.shape
    classes, counts = np.unique(y, return_counts=True)
    pi = counts / counts.sum()
    mu = X.mean(0)
    Sb_diag = np.zeros(d, float)
    for pc, c in zip(pi, classes):
        muc = X[y == c].mean(0)
        diff = muc - mu
        Sb_diag += pc * (diff ** 2)
    return Sb_diag

# ----------------- Base cluster-metric classifier -----------------
class ClusterMetricBase:
    """
    Cluster-based classifier with pluggable metric learning.
    Metric options: 'identity', 'whiten_global', 'whiten_within', 'diag_fisher', 'triplet_diag'
    cluster_mode: 'per_class' (K per class) or 'global' (K overall)
    Predicts by nearest centers in transformed space; also supports soft predict_proba.
    """
    def __init__(
        self,
        metric="whiten_within",
        cluster_mode="per_class",
        n_clusters=5,
        reg=1e-6,
        triplet_iters=1500,
        triplet_lr=0.05,
        triplet_alpha=1e-3,
        feature_subspace_rate=1.0,  # e.g., 0.6 to do random subspace
        random_state=0,
        standardize=True
    ):
        self.metric = metric
        self.cluster_mode = cluster_mode
        self.n_clusters = int(n_clusters)
        self.reg = reg
        self.triplet_iters = int(triplet_iters)
        self.triplet_lr = float(triplet_lr)
        self.triplet_alpha = float(triplet_alpha)
        self.feature_subspace_rate = float(feature_subspace_rate)
        self.random_state = int(random_state)
        self.standardize = standardize

        # learned
        self.scaler_ = None
        self.A_ = None
        self.centers_ = None
        self.center_labels_ = None
        self.classes_ = None
        self.feature_idx_ = None
        self.gamma_ = None  # RBF temperature for proba

    def _select_features(self, d):
        rng = np.random.default_rng(self.random_state)
        m = max(1, int(np.ceil(self.feature_subspace_rate * d)))
        if m >= d:
            return np.arange(d)
        return np.sort(rng.choice(d, size=m, replace=False))

    def _learn_metric(self, X, y):
        n, d = X.shape
        if self.metric == "identity":
            A = np.eye(d)
        elif self.metric == "whiten_global":
            A = _cov_eig_inv_sqrt(X, reg=self.reg)
        elif self.metric == "whiten_within":
            Sw = _pooled_within_scatter(X, y) + self.reg * np.eye(d)
            vals, vecs = eigh(Sw)
            vals = np.clip(vals, self.reg, None)
            A = np.diag(1.0 / np.sqrt(vals)) @ vecs.T
        elif self.metric == "diag_fisher":
            Sw = _pooled_within_scatter(X, y)
            Sw_diag = np.clip(np.diag(Sw), self.reg, None)
            Sb_diag = _between_scatter_diag(X, y)
            w = np.sqrt((Sb_diag + self.reg) / (Sw_diag + self.reg))
            w = w / (np.mean(w) + 1e-12)
            A = np.diag(np.sqrt(np.clip(w, 0.0, None)))
        elif self.metric == "triplet_diag":
            rng = np.random.default_rng(self.random_state)
            w = np.ones(d, float)
            classes = np.unique(y)
            idx_by_c = {c: np.where(y == c)[0] for c in classes}
            for t in range(self.triplet_iters):
                i = int(rng.integers(0, n))
                yi = y[i]
                pos_idx = idx_by_c[yi]
                if pos_idx.size <= 1:
                    continue
                j = int(rng.choice(pos_idx))
                while j == i:
                    j = int(rng.choice(pos_idx))
                neg_c = int(rng.choice([c for c in classes if c != yi]))
                k = int(rng.choice(idx_by_c[neg_c]))
                dp = (X[i] - X[j]) ** 2
                dn = (X[i] - X[k]) ** 2
                dpos = np.dot(w, dp); dneg = np.dot(w, dn)
                margin = 1.0 + dpos - dneg
                if margin > 0:
                    grad = dp - dn + 2 * self.triplet_alpha * w
                else:
                    grad = 2 * self.triplet_alpha * w
                w -= self.triplet_lr * grad
                w = np.maximum(w, 0.0)
                if (t + 1) % 200 == 0:
                    m = np.mean(w[w > 0]) if np.any(w > 0) else 1.0
                    w /= (m + 1e-12)
            A = np.diag(np.sqrt(w))
        else:
            raise ValueError(f"Unknown metric {self.metric}")
        return A

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        n, d = X.shape
        # feature subspace
        self.feature_idx_ = self._select_features(d)
        X = X[:, self.feature_idx_]
        # standardize
        if self.standardize:
            self.scaler_ = StandardScaler()
            Xs = self.scaler_.fit_transform(X)
        else:
            self.scaler_ = None
            Xs = X
        # metric learning
        self.A_ = self._learn_metric(Xs, y)
        Z = Xs @ self.A_.T
        self.classes_ = np.unique(y)

        rng = np.random.RandomState(self.random_state)
        centers, labels = [], []
        if self.cluster_mode == "per_class":
            for c in self.classes_:
                Zc = Z[y == c]
                k = min(self.n_clusters, max(1, Zc.shape[0]))
                km = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
                km.fit(Zc)
                centers.append(km.cluster_centers_)
                labels.append(np.full(k, c))
            self.centers_ = np.vstack(centers)
            self.center_labels_ = np.concatenate(labels)
        elif self.cluster_mode == "global":
            k = min(self.n_clusters, max(1, Z.shape[0]))
            km = KMeans(n_clusters=k, random_state=self.random_state, n_init=10)
            km.fit(Z)
            self.centers_ = km.cluster_centers_
            # majority label per center
            maj = []
            for cid in range(k):
                idx = np.where(km.labels_ == cid)[0]
                if idx.size == 0:
                    maj.append(self.classes_[np.argmax([(y == c).sum() for c in self.classes_])])
                else:
                    vals, cnts = np.unique(y[idx], return_counts=True)
                    maj.append(vals[np.argmax(cnts)])
            self.center_labels_ = np.array(maj)
        else:
            raise ValueError("cluster_mode must be 'per_class' or 'global'")

        # set temperature gamma_ for soft proba (median heuristic)
        if self.centers_.shape[0] >= 2:
            # median pairwise center distance^2
            C = self.centers_
            diffs = C[:, None, :] - C[None, :, :]
            d2 = np.sum(diffs * diffs, axis=2)
            med = np.median(d2[np.triu_indices_from(d2, k=1)])
            self.gamma_ = 1.0 / (med + 1e-8)
        else:
            self.gamma_ = 1.0
        return self

    def _transform(self, X):
        X = np.asarray(X, float)
        X = X[:, self.feature_idx_]
        if self.scaler_ is not None:
            X = self.scaler_.transform(X)
        return X @ self.A_.T

    def predict(self, X):
        proba = self.predict_proba(X)
        # classes_ is sorted; pick argmax
        return self.classes_[np.argmax(proba, axis=1)]

    def predict_proba(self, X):
        Z = self._transform(X)
        z2 = np.sum(Z * Z, axis=1, keepdims=True)              # (n,1)
        c2 = np.sum(self.centers_ * self.centers_, axis=1)     # (k,)
        dot = Z @ self.centers_.T                               # (n,k)
        d2 = z2 + c2[None, :] - 2 * dot                         # (n,k)

        # RBF kernel over centers then sum within each class
        K = np.exp(-self.gamma_ * d2)                           # (n,k)
        # aggregate per class
        classes = self.classes_
        out = np.zeros((Z.shape[0], classes.size), float)
        for j, c in enumerate(classes):
            mask = (self.center_labels_ == c).astype(float)     # (k,)
            out[:, j] = K @ mask
        # normalize to probabilities
        out_sum = out.sum(axis=1, keepdims=True) + 1e-12
        return out / out_sum

# ----------------- Ensemble wrapper -----------------
class ClusterMetricEnsemble:
    """
    Build an ensemble of ClusterMetricBase with diverse metrics/seeds/subspaces.
    Learn nonnegative weights via TimeSeriesSplit CV from balanced accuracy.
    Combine by weighted average of predict_proba.
    """
    def __init__(
        self,
        metrics=("whiten_within", "diag_fisher", "whiten_global", "triplet_diag", "identity"),
        seeds=(0, 13, 21, 37),
        feature_subspace_rates=(1.0, 0.8, 0.6),
        cluster_mode="per_class",
        n_clusters=5,
        tscv_splits=5,
        weight_temp=5.0,          # softmax temperature for converting CV BA to weights
        random_state=0
    ):
        self.metrics = tuple(metrics)
        self.seeds = tuple(seeds)
        self.feature_subspace_rates = tuple(feature_subspace_rates)
        self.cluster_mode = cluster_mode
        self.n_clusters = int(n_clusters)
        self.tscv_splits = int(tscv_splits)
        self.weight_temp = float(weight_temp)
        self.random_state = int(random_state)

        self.base_models_ = []
        self.weights_ = None
        self.classes_ = None

    def _build_base(self, metric, seed, rate):
        return ClusterMetricBase(
            metric=metric,
            cluster_mode=self.cluster_mode,
            n_clusters=self.n_clusters,
            feature_subspace_rate=rate,
            random_state=seed,
            standardize=True
        )

    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        self.classes_ = np.unique(y)

        # Build candidate base models
        configs = []
        for m in self.metrics:
            for s in self.seeds:
                for r in self.feature_subspace_rates:
                    configs.append((m, s, r))

        # TimeSeries CV to score each config
        tscv = TimeSeriesSplit(n_splits=self.tscv_splits)
        cv_scores = np.zeros(len(configs), float)

        for idx, (m, s, r) in enumerate(configs):
            scores = []
            for tr_idx, va_idx in tscv.split(X):
                Xtr, ytr = X[tr_idx], y[tr_idx]
                Xva, yva = X[va_idx], y[va_idx]
                model = self._build_base(m, s, r).fit(Xtr, ytr)
                yhat = model.predict(Xva)
                scores.append(balanced_accuracy_score(yva, yhat))
            cv_scores[idx] = np.mean(scores)

        # Convert scores to weights via softmax
        # (nonnegative, emphasize better configs with temperature)
        logits = self.weight_temp * (cv_scores - np.max(cv_scores))
        w = np.exp(logits)
        w = w / (w.sum() + 1e-12)
        self.weights_ = w

        # Fit all bases on full train and store
        self.base_models_ = [self._build_base(*cfg).fit(X, y) for cfg in configs]
        return self

    def predict_proba(self, X):
        # Weighted average of probabilities
        Ps = []
        for model in self.base_models_:
            Ps.append(model.predict_proba(X))
        Ps = np.stack(Ps, axis=2)  # (n, C, B)
        w = self.weights_[None, None, :]  # (1,1,B)
        P = np.sum(Ps * w, axis=2)
        # guard: renormalize
        P /= (P.sum(axis=1, keepdims=True) + 1e-12)
        return P

    def predict(self, X):
        P = self.predict_proba(X)
        return self.classes_[np.argmax(P, axis=1)]

    def score_balanced_accuracy(self, X, y):
        y_pred = self.predict(X)
        return balanced_accuracy_score(y, y_pred)

# ----------------- Example run (n_clusters = 5) -----------------
# Assume X_train, y_train, X_test, y_test exist (y_* shape (n_samples,))
# ens = ClusterMetricEnsemble(
#     metrics=("whiten_within", "diag_fisher", "whiten_global", "triplet_diag", "identity"),
#     seeds=(0, 13, 21, 37),
#     feature_subspace_rates=(1.0, 0.8, 0.6),
#     cluster_mode="per_class",
#     n_clusters=5,
#     tscv_splits=5,
#     weight_temp=6.0
# )
# ens.fit(X_train, y_train)
# print("Ensemble weights (sum=1):", ens.weights_)
# print("BA train:", ens.score_balanced_accuracy(X_train, y_train))
# print("BA test :", ens.score_balanced_accuracy(X_test,  y_test))


In [19]:
bacc = []
for seed in range(30):
    pack = generate_data(seed=seed)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    ens = ClusterMetricEnsemble(
        metrics=("whiten_within", "diag_fisher", "whiten_global", "triplet_diag", "identity"),
        seeds=(0, 13, 21, 37),
        feature_subspace_rates=(1.0, 0.8, 0.6),
        cluster_mode="per_class",
        n_clusters=5,
        tscv_splits=5,
        weight_temp=6.0
    )
    ens.fit(X_train, y_train)
    test_bacc = ens.score_balanced_accuracy(X_test, y_test)
    bacc.append(test_bacc)

print(f"Test BA: {np.mean(bacc):.4f} ± {np.std(bacc):.4f}")

Test BA: 0.6826 ± 0.0221
